# 12.6 - CI/CD for AI Apps

**Module 12 - Production Deployment** | Netsetos GenAI Engineering

This notebook works through CI/CD for AI Apps hands-on: Why AI CI/CD is different; The pipeline is a graph of gates; The eval gate: prompt regression under non-determinism; Output validators: guard every response; The GitHub Actions workflow and caching; Keyless deploy with OIDC.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Every cell in this notebook is a self-contained simulation that runs with plain Python and no network - so this first cell only needs one optional helper for loading keys from a `.env` file. It is commented out; uncomment it on Colab or a fresh environment.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


**Explanation**

A one-line dependency stub. `python-dotenv` is only needed if you want to read keys from a `.env` file; nothing else here requires an install.

**How the code works - step by step**
- **`# !pip install python-dotenv -q`** - commented out; uncomment on a fresh env to install the only external helper.

**In one line:** optional dependency prep - no install needed to run the simulations.

**What you'll see:** (no output - environment prep)

## Setup - API keys

Real AI CI/CD reads provider keys from the environment, never from hardcoded strings. This cell sets up empty placeholders so the code has somewhere to look - but note every cell in this lesson is a keyless simulation, so you can run all of them with these left blank.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

An environment-variable scaffold, not a live API call. `os.environ.setdefault` only writes a key if it is not already set, so a real key you exported beforehand is preserved while a missing one becomes an empty string.

**How the code works - step by step**
- **`import os`** - the standard library module for reading and writing environment variables.
- **`os.environ.setdefault("OPENAI_API_KEY", "")`** - seeds an empty OpenAI key only if none exists.
- **`os.environ.setdefault("ANTHROPIC_API_KEY", "")`** - same for Anthropic (this is the one a real eval gate's LLM-judge would use).
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - same for Google.

**In one line:** load keys from the environment, never hardcode them - though nothing here actually calls a provider.

**What you'll see:** (no output - environment prep)

## 1 - Why AI CI/CD is different

Start with the problem the whole lesson solves: a normal unit test checks that code returns the *right value*, but an LLM app can return a *worse answer* while every assertion stays green. This cell puts a deterministic test next to a quality-judged one so you can watch a prompt edit pass the test and still regress.

In [ ]:
# A traditional unit test is deterministic; an LLM answer is judged on QUALITY.
def unit_test(fn, x, expected):
    return "PASS" if fn(x) == expected else "FAIL"

def add(x):
    return x + x

# The AI app: a prompt version maps to a scripted answer + a judge quality score (1-10).
ANSWERS = {
    "v1 (be concise)":            {"text": "Take the GenAI Engineering course - it covers agents in depth.", "score": 8.6},
    "v2 (be brief and friendly)": {"text": "Oh hey, great question! You should totally check out our GenAI course, it is amazing!", "score": 5.9},
}
def keyword_test(answer):     # a naive "unit test" on the AI output
    return "PASS" if "GenAI" in answer["text"] else "FAIL"

print("Traditional unit test (deterministic - same result every run):")
for r in range(3):
    print("  run {}: add(2) == 4 -> {}".format(r + 1, unit_test(add, 2, 4)))
print()

print("Prompt edit v1 -> v2 (a 'friendlier' tweak), same keyword unit test:")
for v, a in ANSWERS.items():
    print("  {:<28} keyword_test={}   judge_quality={}/10".format(v, keyword_test(a), a["score"]))
print()
print("The keyword unit test stays GREEN on both versions, but the judge score dropped 8.6 -> 5.9.")
print("A prompt edit silently REGRESSED quality - only an eval gate catches it.")

# Output:
# Traditional unit test (deterministic - same result every run):
#   run 1: add(2) == 4 -> PASS
#   run 2: add(2) == 4 -> PASS
#   run 3: add(2) == 4 -> PASS
#
# Prompt edit v1 -> v2 (a 'friendlier' tweak), same keyword unit test:
#   v1 (be concise)              keyword_test=PASS   judge_quality=8.6/10
#   v2 (be brief and friendly)   keyword_test=PASS   judge_quality=5.9/10
#
# The keyword unit test stays GREEN on both versions, but the judge score dropped 8.6 -> 5.9.
# A prompt edit silently REGRESSED quality - only an eval gate catches it.

**Explanation**

A side-by-side contrast, not a model call. It runs a genuinely deterministic unit test three times to show it never varies, then applies a naive keyword test to two scripted prompt versions whose real difference lives in a separate judge quality score. The point is that the cheap assertion cannot see the regression the score reveals.

**How the code works - step by step**
- **`unit_test`** - returns `"PASS"`/`"FAIL"` by comparing `fn(x)` to an expected value; deterministic by construction.
- **`add`** - a trivial function (`x + x`) used to demonstrate a test that gives the same verdict every run.
- **`ANSWERS`** - maps two prompt versions (`v1 (be concise)`, `v2 (be brief and friendly)`) to a scripted `text` plus a `score` (8.6 vs 5.9).
- **`keyword_test`** - a naive AI 'unit test' that only checks whether `"GenAI"` appears in the text.
- **the loops** - print `add(2) == 4` three times (all PASS), then print each version's keyword result (both PASS) alongside its judge score.

**In one line:** the keyword test stays green on both versions while the judge score drops 8.6 -> 5.9 - a silent quality regression only an eval gate catches.

**What you'll see:** Three identical `PASS` runs of the deterministic unit test, then both prompt versions showing `keyword_test=PASS` while the judge score falls from 8.6/10 to 5.9/10, with a closing note that only an eval gate would catch the regression.

## 2 - The pipeline is a graph of gates

Zoom out to the shape of the whole system. CI/CD is one automated chain - lint, tests, eval, build, scan, deploy, canary - run on every push or PR, where a failed gate stops the line and the cheap gates go first. This cell runs a commit down that chain twice to show both behaviours.

In [ ]:
# A CI/CD pipeline is an ordered chain of GATES; a failed gate STOPS the line.
GATES = ["lint", "unit-tests", "eval-gate", "build (once, SHA-tagged)", "scan",
         "deploy-staging", "smoke-test", "canary", "prod"]

def run_pipeline(failing_gate=None):
    image = None
    for g in GATES:
        if failing_gate and g.startswith(failing_gate):
            print("  [FAIL] " + g + "   <- line STOPS here; nothing downstream runs")
            return False
        note = ""
        if g.startswith("build"):
            image = "ai-service:a1b2c3d"; note = "   built " + image
        elif g.startswith("deploy") or g in ("canary", "prod"):
            note = "   deploy " + str(image) + " (same artifact)"
        print("  [PASS] " + g + note)
    return True

print("Commit 1 - every gate passes:")
run_pipeline()
print()
print("Commit 2 - the eval gate fails (a prompt regression):")
run_pipeline(failing_gate="eval-gate")
print()
print("Run the cheapest gates first so a bad change fails fast; one failed gate blocks the deploy.")
print("The image is built ONCE and the SAME artifact flows through staging -> canary -> prod.")

# Output:
# Commit 1 - every gate passes:
#   [PASS] lint
#   [PASS] unit-tests
#   [PASS] eval-gate
#   [PASS] build (once, SHA-tagged)   built ai-service:a1b2c3d
#   [PASS] scan
#   [PASS] deploy-staging   deploy ai-service:a1b2c3d (same artifact)
#   [PASS] smoke-test
#   [PASS] canary   deploy ai-service:a1b2c3d (same artifact)
#   [PASS] prod   deploy ai-service:a1b2c3d (same artifact)
#
# Commit 2 - the eval gate fails (a prompt regression):
#   [PASS] lint
#   [PASS] unit-tests
#   [FAIL] eval-gate   <- line STOPS here; nothing downstream runs
#
# Run the cheapest gates first so a bad change fails fast; one failed gate blocks the deploy.
# The image is built ONCE and the SAME artifact flows through staging -> canary -> prod.

**Explanation**

A tiny pipeline runner that models the two rules that make gates work: a failure halts everything downstream, and the image is built exactly once and promoted unchanged. `run_pipeline` walks an ordered list and short-circuits at the named failing gate.

**How the code works - step by step**
- **`GATES`** - the ordered stages from `lint` through `unit-tests`, `eval-gate`, a single SHA-tagged `build`, `scan`, `deploy-staging`, `smoke-test`, `canary`, `prod`.
- **`run_pipeline(failing_gate=None)`** - iterates the gates; when a gate name matches `failing_gate` it prints `[FAIL]` and returns immediately, so nothing after it runs.
- **the `build` branch** - sets `image = "ai-service:a1b2c3d"` once; **the deploy/canary/prod branches** reuse that same `image`, proving one artifact flows through every stage.
- **the two calls** - `run_pipeline()` passes everything; `run_pipeline(failing_gate="eval-gate")` stops at the eval gate.

**In one line:** cheapest gates first, one failed gate blocks the deploy, and the same image is built once and promoted staging -> canary -> prod.

**What you'll see:** Commit 1 prints `[PASS]` down the full chain with `ai-service:a1b2c3d` built once and reused for every deploy; commit 2 prints `[PASS]` for lint and unit-tests, then `[FAIL] eval-gate` with a note that the line stops and nothing downstream runs.

## 3 - The eval gate: prompt regression under non-determinism

This is the AI-specific gate and the heart of the lesson. You keep a suite of prompt tests scored by an LLM-as-judge, and a change may merge only if it keeps the pass rate above a threshold - but because the judge is non-deterministic, you repeat each test and pass by majority rather than exact match. This cell runs a suite before and after a prompt edit.

> **No API key needed** - the judge scores are pre-scripted so the gate logic runs offline.

In [ ]:
# Each prompt test is scored by the judge 3 times (LLM-as-judge is non-deterministic).
# A test passes if a MAJORITY of runs clear the bar; the GATE needs the overall PASS RATE
# to stay at or above a threshold.
JUDGE_BAR = 7      # a single judge run passes at score >= 7
GATE = 0.80        # the suite must keep >= 80% of tests passing

def test_passes(scores):            # majority of 3 runs at or above the bar
    return sum(1 for s in scores if s >= JUDGE_BAR) >= 2

def run_suite(name, suite):
    passed = 0
    print(name + ":")
    for t, scores in suite.items():
        ok = test_passes(scores)
        passed += ok
        print("  {:<16} judge runs {} -> {}".format(t, scores, "PASS" if ok else "FAIL"))
    rate = passed / len(suite)
    verdict = "GATE PASS" if rate >= GATE else "GATE FAIL - block the merge"
    print("  pass rate {}/{} = {:.0%}  (need {:.0%})  -> {}\n".format(passed, len(suite), rate, GATE, verdict))
    return rate

# Baseline on main: 9 of 10 pass. The flaky "summarize" [7,4,7] still passes by majority.
baseline = {
    "greeting": [9, 8, 9], "refund_policy": [8, 8, 7], "safety_refusal": [9, 9, 8],
    "json_output": [7, 8, 8], "summarize": [7, 4, 7], "translate": [8, 7, 8],
    "code_help": [8, 9, 8], "recommend": [7, 7, 8], "tone": [8, 8, 7], "edge_case": [4, 3, 4],
}
run_suite("Baseline (main branch)", baseline)

# After a prompt edit several tests regress below the bar.
edited = dict(baseline)
edited.update({"greeting": [5, 6, 5], "refund_policy": [4, 5, 4], "summarize": [3, 4, 3], "tone": [5, 4, 5]})
run_suite("After the prompt edit (this PR)", edited)

# Output:
# Baseline (main branch):
#   greeting         judge runs [9, 8, 9] -> PASS
#   refund_policy    judge runs [8, 8, 7] -> PASS
#   safety_refusal   judge runs [9, 9, 8] -> PASS
#   json_output      judge runs [7, 8, 8] -> PASS
#   summarize        judge runs [7, 4, 7] -> PASS
#   translate        judge runs [8, 7, 8] -> PASS
#   code_help        judge runs [8, 9, 8] -> PASS
#   recommend        judge runs [7, 7, 8] -> PASS
#   tone             judge runs [8, 8, 7] -> PASS
#   edge_case        judge runs [4, 3, 4] -> FAIL
#   pass rate 9/10 = 90%  (need 80%)  -> GATE PASS
#
# After the prompt edit (this PR):
#   greeting         judge runs [5, 6, 5] -> FAIL
#   refund_policy    judge runs [4, 5, 4] -> FAIL
#   safety_refusal   judge runs [9, 9, 8] -> PASS
#   json_output      judge runs [7, 8, 8] -> PASS
#   summarize        judge runs [3, 4, 3] -> FAIL
#   translate        judge runs [8, 7, 8] -> PASS
#   code_help        judge runs [8, 9, 8] -> PASS
#   recommend        judge runs [7, 7, 8] -> PASS
#   tone             judge runs [5, 4, 5] -> FAIL
#   edge_case        judge runs [4, 3, 4] -> FAIL
#   pass rate 5/10 = 50%  (need 80%)  -> GATE FAIL - block the merge

**Explanation**

A gate simulator that separates two levels of tolerance: each test tolerates a flaky judge via majority-of-3, and the suite tolerates a few hard failures via a pass-rate threshold. The scripted scores let you see both the majority rule and the threshold decide a merge.

**How the code works - step by step**
- **`JUDGE_BAR = 7` / `GATE = 0.80`** - a single judge run passes at score >= 7; the suite must keep >= 80% of tests passing.
- **`test_passes(scores)`** - returns True when at least 2 of 3 judge runs clear the bar (majority vote), so a flaky `[7, 4, 7]` still passes.
- **`run_suite(name, suite)`** - scores every test, prints its three judge runs and PASS/FAIL, then computes the pass rate and prints `GATE PASS` or `GATE FAIL - block the merge`.
- **`baseline`** - 10 tests where 9 pass (only `edge_case` fails) -> 90%.
- **`edited`** - copies the baseline then regresses `greeting`, `refund_policy`, `summarize`, `tone` below the bar -> 50%.

**In one line:** score a suite with a repeated majority-vote judge, gate on the pass rate, and block any change that drops it below the threshold.

**What you'll see:** The baseline suite prints 9 PASS / 1 FAIL for a 90% pass rate and `GATE PASS`; after the prompt edit four more tests flip to FAIL, dropping the rate to 50% and printing `GATE FAIL - block the merge`.

## 4 - Output validators: guard every response

The eval gate is powerful but expensive, so you cannot run it on every production request. Output validators fill that gap with cheap deterministic checks - empty, repetitive, PII leak, wrong language - that run in CI on test prompts *and* in production on every real response. This cell runs the validators over a few sample outputs.

In [ ]:
import re
# Output validators: cheap DETERMINISTIC guards. Run in CI on test prompts AND in
# production on every response, before it reaches the user.
def validate(output, expected_language="en"):
    issues = []
    if not output or len(output.strip()) < 5:
        issues.append(("critical", "empty or too short"))
    sents = [s.strip() for s in output.split(".") if len(s.strip()) > 10]
    if len(sents) > 4:
        uniq = len(set(s.lower() for s in sents))
        if uniq < len(sents) * 0.5:
            issues.append(("high", "repetitive: {} unique of {} sentences".format(uniq, len(sents))))
    pii = {"email": r"\b[\w.%+-]+@[\w.-]+\.[A-Za-z]{2,}\b", "phone": r"\b(?:\+91[\s-]?)?[6-9]\d{9}\b"}
    for kind, pat in pii.items():
        if re.search(pat, output):
            issues.append(("critical", "PII leak: {} pattern".format(kind)))
    if expected_language == "en":
        non_ascii = sum(1 for c in output if ord(c) > 127) / max(len(output), 1)
        if non_ascii > 0.3:
            issues.append(("medium", "expected English but {:.0%} non-ASCII".format(non_ascii)))
    critical = any(sev == "critical" for sev, _ in issues)
    return issues, critical

OUTPUTS = [
    "MCP is an open standard by Anthropic for connecting AI models to tools and data.",
    "",
    "The answer is 42. The answer is 42. The answer is 42. The answer is 42. The answer is 42.",
    "Sure, reach our team at admin@netsetos.com or call +91 9876543210 any time.",
]
print("Output validators - deterministic, keyless, run in CI and in production:")
for o in OUTPUTS:
    issues, critical = validate(o)
    label = o[:44] + ("..." if len(o) > 44 else "")
    verdict = "PASS " if not issues else ("BLOCK" if critical else "WARN ")
    print('  [{}] "{}"'.format(verdict, label))
    for sev, msg in issues:
        print("          - [{}] {}".format(sev, msg))

# Output:
# Output validators - deterministic, keyless, run in CI and in production:
#   [PASS ] "MCP is an open standard by Anthropic for con..."
#   [BLOCK] ""
#           - [critical] empty or too short
#   [WARN ] "The answer is 42. The answer is 42. The answ..."
#           - [high] repetitive: 1 unique of 5 sentences
#   [BLOCK] "Sure, reach our team at admin@netsetos.com o..."
#           - [critical] PII leak: email pattern
#           - [critical] PII leak: phone pattern

**Explanation**

A pure-function validator that returns a list of severity-tagged issues plus a critical flag; no model, no network, just string and regex checks. Read it as the fast bouncer that stands at the door of every response, in test and in live traffic alike.

**How the code works - step by step**
- **`validate(output, expected_language="en")`** - accumulates `issues` and decides if any is `critical`.
- **empty check** - flags `critical` when the trimmed output is under 5 characters.
- **repetition check** - splits into sentences and flags `high` when fewer than half are unique (a stuck loop).
- **PII check** - regexes for `email` and Indian `phone` patterns, each flagged `critical` on a match.
- **language check** - flags `medium` when more than 30% of characters are non-ASCII but English was expected.
- **the loop** - runs four sample outputs and prints `PASS`/`WARN`/`BLOCK` (BLOCK when any issue is critical) with each issue listed.

**In one line:** cheap deterministic guards that block empty and PII-leaking responses and warn on repetition - fast enough to run on every live response.

**What you'll see:** The clean MCP sentence prints `[PASS ]`; the empty string prints `[BLOCK]` (empty or too short); the repeated sentence prints `[WARN ]` (repetitive: 1 unique of 5); and the contact-info line prints `[BLOCK]` with both email and phone PII-leak issues.

## 5 - The GitHub Actions workflow and caching

Now the concrete tool: a GitHub workflow is YAML that runs jobs of steps on every push or PR, and caching unchanged dependencies and Docker layers is the single biggest speedup. A matrix runs configurations in parallel. This cell models a run cold-versus-cached and a parallel matrix so you can see where the time goes.

In [ ]:
# A GitHub Actions run is a list of steps, each with a cold and a cached duration (seconds).
STEPS = [
    ("checkout",        3,   3),
    ("setup deps (uv)", 48,  4),    # the big win: cached dependencies
    ("lint",            6,   6),
    ("unit tests",      22,  22),
    ("eval gate",       35,  35),
    ("docker build",    70,  9),    # cached layers
    ("scan (trivy)",    12,  12),
]
cold = sum(c for _, c, _ in STEPS)
warm = sum(w for _, _, w in STEPS)
print("A workflow run - cold cache vs warm cache:")
print("  {:<18}{:>7}{:>9}".format("step", "cold", "cached"))
for name, c, w in STEPS:
    print("  {:<18}{:>6}s{:>8}s".format(name, c, w))
print("  {:<18}{:>6}s{:>8}s   -> {:.0%} faster with the cache".format("TOTAL", cold, warm, 1 - warm / cold))
print()

# A matrix runs configs in PARALLEL, so wall-clock is the slowest lane, not the sum.
MATRIX = {"py3.12": warm, "py3.13": warm + 3}
serial = sum(MATRIX.values())
parallel = max(MATRIX.values())
print("A matrix (Python 3.12 + 3.13) runs in parallel:")
for k, v in MATRIX.items():
    print("  {:<8} {}s".format(k, v))
print("  serial would be {}s; parallel wall-clock is {}s (the slowest lane).".format(serial, parallel))

# Output:
# A workflow run - cold cache vs warm cache:
#   step                 cold   cached
#   checkout               3s       3s
#   setup deps (uv)       48s       4s
#   lint                   6s       6s
#   unit tests            22s      22s
#   eval gate             35s      35s
#   docker build          70s       9s
#   scan (trivy)          12s      12s
#   TOTAL                196s      91s   -> 54% faster with the cache
#
# A matrix (Python 3.12 + 3.13) runs in parallel:
#   py3.12   91s
#   py3.13   94s
#   serial would be 185s; parallel wall-clock is 94s (the slowest lane).

**Explanation**

A timing model, not a real CI run. Each step carries a cold and a cached duration; summing both columns shows how much caching saves, and a two-entry matrix shows that parallel wall-clock is the slowest lane rather than the sum. It also quietly exposes the one step caching cannot help.

**How the code works - step by step**
- **`STEPS`** - each `(name, cold, cached)` tuple; `setup deps (uv)` (48->4) and `docker build` (70->9) are the big wins, while `unit tests` and `eval gate` are identical cold and cached.
- **`cold` / `warm`** - sum the two columns and print a per-step table plus the total and percent saved.
- **`MATRIX`** - two Python versions each taking about the warm total; **`serial`** sums them, **`parallel`** takes the `max`.
- **the prints** - show the ~54% cache speedup and that testing two versions costs the slower lane, not the sum.

**In one line:** cache deps and Docker layers to roughly halve the run, and use a matrix to test two configs for the wall-clock price of one.

**What you'll see:** A cold-vs-cached step table totalling 196s -> 91s (54% faster with the cache), then a matrix showing py3.12 at 91s and py3.13 at 94s, with serial 185s but parallel wall-clock 94s - and the eval gate visibly unchanged at 35s because it spends real judge tokens every run.

## 6 - Keyless deploy with OIDC

To deploy, the CI job must prove to the cloud that it is allowed to. The old way is a stored service-account JSON key that never expires and is a disaster if it leaks; the 2026 best practice is OIDC keyless auth, where GitHub mints a short-lived token the cloud verifies against an attribute condition. This cell contrasts the two.

In [ ]:
# Two ways for the CI job to authenticate to the cloud in order to deploy.

# (A) A stored service-account JSON key: a long-lived secret sitting in repo settings.
def stored_key(leaked):
    return "stored JSON key: " + ("COMPROMISED - valid until someone manually rotates it" if leaked
                                   else "works, but never expires on its own")

print("Approach A - a stored service-account key (the anti-pattern):")
print("  " + stored_key(leaked=False))
print("  " + stored_key(leaked=True))
print()

# (B) OIDC: GitHub mints a short-lived token with claims (repo, ref); the cloud verifies it
# against an attribute condition (which repo + branch may deploy), then grants a ~1-hour credential.
ALLOW = {"repo": "netsetos/ai-service", "ref": "refs/heads/main"}
def oidc(repo, ref, age_min):
    if repo != ALLOW["repo"] or ref != ALLOW["ref"]:
        return "DENIED by the attribute condition (wrong repo/branch)"
    if age_min > 60:
        return "DENIED - token expired ({} min > 60)".format(age_min)
    return "ALLOWED - short-lived token, no stored secret"

print("Approach B - OIDC keyless (Workload Identity Federation):")
print("  main branch,   5 min old  -> " + oidc("netsetos/ai-service", "refs/heads/main", 5))
print("  a fork,        5 min old  -> " + oidc("attacker/ai-service", "refs/heads/main", 5))
print("  main branch,  75 min old  -> " + oidc("netsetos/ai-service", "refs/heads/main", 75))
print()
print("OIDC: nothing stored to leak or rotate; the token is scoped to your repo/branch and expires in about an hour.")

# Output:
# Approach A - a stored service-account key (the anti-pattern):
#   stored JSON key: works, but never expires on its own
#   stored JSON key: COMPROMISED - valid until someone manually rotates it
#
# Approach B - OIDC keyless (Workload Identity Federation):
#   main branch,   5 min old  -> ALLOWED - short-lived token, no stored secret
#   a fork,        5 min old  -> DENIED by the attribute condition (wrong repo/branch)
#   main branch,  75 min old  -> DENIED - token expired (75 min > 60)
#
# OIDC: nothing stored to leak or rotate; the token is scoped to your repo/branch and expires in about an hour.

**Explanation**

A two-approach comparison that makes the security difference concrete. The stored-key function shows a credential that stays valid until manually rotated; the OIDC function enforces both an attribute condition (which repo and branch may deploy) and a token lifetime, denying a fork and an expired token.

**How the code works - step by step**
- **`stored_key(leaked)`** - returns a string showing a JSON key that either 'never expires on its own' or, if leaked, is 'COMPROMISED - valid until someone manually rotates it'.
- **`ALLOW`** - the attribute condition: only `netsetos/ai-service` on `refs/heads/main` may deploy.
- **`oidc(repo, ref, age_min)`** - denies a repo/branch mismatch, denies a token older than 60 minutes, otherwise allows with no stored secret.
- **the three OIDC calls** - main branch fresh (ALLOWED), a fork (DENIED by the attribute condition), main branch 75 min old (DENIED - expired).

**In one line:** OIDC mints a short-lived token scoped to your repo and branch, so there is nothing stored to leak or rotate and a fork or expired token is denied.

**What you'll see:** Approach A prints a stored key that works but never expires and, when leaked, stays valid until manual rotation; Approach B prints ALLOWED for the fresh main-branch token, DENIED by the attribute condition for the fork, and DENIED as expired for the 75-minute-old token.

## 7 - Canary and rollback

The pipeline passed every gate; now ship it *safely*. Deploying straight to 100% of traffic means a bug that slipped every gate hits every user at once, so a canary deploys the new revision at zero traffic and shifts it 10 -> 50 -> 100 while watching the error rate - rolling back to the previous revision in seconds if it spikes. This cell runs a healthy and a bad canary.

In [ ]:
# Deploy the new revision at 0% traffic, then shift gradually while watching the error rate.
# Roll back to the previous revision in seconds if the canary misbehaves.
ERR_THRESHOLD = 2.0    # percent; above this, roll back

def rollout(name, canary_errors):
    print(name + ":")
    print("  deploy rev-new --no-traffic --tag=canary    (rev-old 100%, rev-new 0%)")
    for pct, err in zip([10, 50, 100], canary_errors):
        line = "  shift rev-new to {:>3}%   canary error rate {:.1f}%".format(pct, err)
        if err > ERR_THRESHOLD:
            print(line + "   -> SPIKE > {:.0f}%: ROLLBACK to rev-old 100% (seconds)".format(ERR_THRESHOLD))
            print("  final: rev-old 100%, rev-new 0% - bad canary caught before full rollout\n")
            return
        print(line + "   ok")
    print("  final: rev-new 100% - promoted; rev-old kept for an instant rollback\n")

rollout("Scenario A - a healthy canary", [0.4, 0.5, 0.6])
rollout("Scenario B - a bad canary (a regression slipped through)", [0.5, 6.3, 0.0])
print("Canary = ship to a slice, watch the error rate, promote or roll back in seconds - never straight to 100%.")

# Output:
# Scenario A - a healthy canary:
#   deploy rev-new --no-traffic --tag=canary    (rev-old 100%, rev-new 0%)
#   shift rev-new to  10%   canary error rate 0.4%   ok
#   shift rev-new to  50%   canary error rate 0.5%   ok
#   shift rev-new to 100%   canary error rate 0.6%   ok
#   final: rev-new 100% - promoted; rev-old kept for an instant rollback
#
# Scenario B - a bad canary (a regression slipped through):
#   deploy rev-new --no-traffic --tag=canary    (rev-old 100%, rev-new 0%)
#   shift rev-new to  10%   canary error rate 0.5%   ok
#   shift rev-new to  50%   canary error rate 6.3%   -> SPIKE > 2%: ROLLBACK to rev-old 100% (seconds)
#   final: rev-old 100%, rev-new 0% - bad canary caught before full rollout
#
# Canary = ship to a slice, watch the error rate, promote or roll back in seconds - never straight to 100%.

**Explanation**

A rollout simulator that models the blast-radius argument: it steps traffic up while checking each stage's error rate against a threshold, and on a spike it stops and reverts to the old revision. Because the previous revision is still live, the rollback is instant.

**How the code works - step by step**
- **`ERR_THRESHOLD = 2.0`** - the error-rate percent above which the canary rolls back.
- **`rollout(name, canary_errors)`** - prints the initial zero-traffic deploy (`--no-traffic --tag=canary`), then walks the `[10, 50, 100]` traffic steps paired with the given error rates.
- **the spike branch** - when a step's error exceeds the threshold, it prints `ROLLBACK to rev-old 100% (seconds)` and returns before reaching full traffic.
- **the two scenarios** - a healthy canary `[0.4, 0.5, 0.6]` promotes to 100%; a bad canary `[0.5, 6.3, 0.0]` spikes at the 50% step and rolls back.

**In one line:** ship to a small traffic slice, watch the error rate, then promote or roll back to the previous revision in seconds - never straight to 100%.

**What you'll see:** Scenario A shifts the new revision 10 -> 50 -> 100% with low error rates and promotes it; Scenario B shifts to 10% fine, then spikes to 6.3% at 50%, triggering an immediate rollback to rev-old at 100% so the bad canary is caught before full rollout.

You built the whole AI delivery pipeline as seven small runnable models: a keyword unit test that stays green while judge quality drops (why AI CI/CD exists), the ordered chain of gates that fails fast and builds once, the eval gate that survives a non-deterministic judge with repeats and majority voting, the cheap deterministic validators that guard every response in test and in production, the caching and matrix that make a GitHub Actions run fast, OIDC keyless auth that removes the stored cloud key, and a canary you can roll back in seconds. Read them together as the answer to five questions of any AI pipeline: does it fail fast, build once, gate on quality, deploy without stored keys, and roll back in seconds. From here, Lesson 14.4 builds the eval set the gate runs, 12.8 wires up the monitoring the canary reads, and 12.7 scales the deployed service under load.